# 🧠 EX47: Confidence Score in Object Detection
### *A Lecture by your Computer Vision Professor*

Welcome back, class! Today, we are going to dive deep into one of the most critical elements of object detection: **Confidence Scores**.

In this lecture, we will:
1. Break down the **Classic YOLO (v1-v5)** anchor-based confidence formula.
2. Explore the modern **Anchor-Free YOLO (v8-v11)** task-aligned assigner score.
3. Understand how varying the confidence threshold (`conf`) shifts **Precision, Recall, True Positives (TP), False Positives (FP), and False Negatives (FN)**.
4. Implement these formulas from scratch and visualize their behavior.

Let's begin!

---

## 📖 1. The Anatomy of a Confidence Score

### A. Classic YOLO (v1 - v5)
In the classic anchor-based architecture, the network predicts bounding boxes and a separate "objectness" probability. The final class-specific confidence score is formulated as:

$$\text{Class Score} = P(\text{Class}_i | \text{Object}) \times P(\text{Object}) \times \text{IoU}_{\text{pred}}^{\text{truth}}$$

Where:
- $P(\text{Class}_i | \text{Object})$: The conditional probability that the object belongs to class $i$, given that there is an object.
- $P(\text{Object})$: The **objectness score** (probability that the grid cell/anchor contains *any* object).
- $\text{IoU}_{\text{pred}}^{\text{truth}}$: The Intersection over Union between the predicted box and the ground truth box.

This three-way multiplication ensures that a detection is only highly confident if the network is sure there's an object, sure of its class, and has localized it accurately.

### B. Anchor-Free YOLO (v8 - v11)
Modern YOLO architectures (v8, v9, v10, v11) discard the separate objectness branch entirely to decrease latency and simplify the loss landscape. Instead of separate heads, they directly predict class probabilities ($s$) for each anchor. During training and assignment, they align classification and localization quality into a single score $t$ using the **Task-Aligned Assigner**:

$$t = s^\alpha \times \text{IoU}^\beta$$

Where:
- $s$ is the predicted class probability.
- $\text{IoU}$ is the spatial overlap with the ground-truth box.
- $\alpha$ and $\beta$ are weighting factors that control the influence of classification vs localization. In Ultralytics YOLOv8/v11, the defaults are $\alpha = 0.5$ and $\beta = 6.0$.

The high value of $\beta=6.0$ heavily penalizes predictions with low overlap, forcing the model to prioritize bounding box precision.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calculate_classic_confidence(p_class_given_obj, p_obj, iou):
    """Calculates classic YOLO class score: P(Class|Obj) * P(Obj) * IoU"""
    return p_class_given_obj * p_obj * iou

def calculate_task_aligned_score(s, iou, alpha=0.5, beta=6.0):
    """Calculates task-aligned score: s^alpha * IoU^beta"""
    return (s ** alpha) * (iou ** beta)

## 🔬 2. Shifting Thresholds: The Precision-Recall Trade-off

Let's recall the definitions of our core metrics from first principles:
- **True Positive (TP)**: A prediction that correctly identifies a ground-truth object (Confidence $\ge$ Threshold and $\text{IoU} \ge \text{IoU\_Threshold}$).
- **False Positive (FP)**: An incorrect prediction (Confidence $\ge$ Threshold and $\text{IoU} < \text{IoU\_Threshold}$).
- **False Negative (FN)**: A ground-truth object that was *not* detected (no prediction matched it with Confidence $\ge$ Threshold and $\text{IoU} \ge \text{IoU\_Threshold}$).

### Mathematical Definitions:
$$\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}$$
*Out of all predicted objects, how many were correct?*

$$\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}} = \frac{\text{TP}}{\text{Total Ground Truths}}$$
*Out of all actual objects in the image, how many did we find?*

### Shifting the Threshold (\tau):
- **High Threshold (e.g. 0.8)**: Very strict. We only keep predictions that the network is highly confident in. This decreases False Positives (high Precision), but also causes us to miss many objects (low Recall, high False Negatives).
- **Low Threshold (e.g. 0.1)**: Very lenient. We keep almost everything. This increases True Positives (high Recall, low False Negatives), but introduces many false alarms (low Precision, high Positives).

In [ ]:
def evaluate_predictions(predictions, num_ground_truths, conf_threshold, iou_threshold=0.5):
    """Evaluates TP, FP, FN, Precision, and Recall at a specific threshold."""
    filtered_preds = [p for p in predictions if p['score'] >= conf_threshold]
    
    tp = sum(1 for p in filtered_preds if p['iou'] >= iou_threshold)
    fp = sum(1 for p in filtered_preds if p['iou'] < iou_threshold)
    fn = num_ground_truths - tp
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0  # Standard choice when no predictions are made
    recall = tp / num_ground_truths if num_ground_truths > 0 else 0.0
    
    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall
    }

## 📊 3. Simulating and Visualizing the Threshold Shift

Let's simulate a dataset of 120 predictions against 60 ground-truth objects. We will vary the threshold from 0.0 to 1.0 and observe how the metrics shift.

In [ ]:
np.random.seed(42)
num_gt = 60

# Simulate 120 predictions
# Scores uniformly distributed
scores = np.random.uniform(0.05, 0.99, 120)
# IoUs around a bell curve centered at 0.55
ious = np.clip(np.random.normal(0.55, 0.22, 120), 0.0, 1.0)

simulated_predictions = [{"score": s, "iou": i} for s, i in zip(scores, ious)]

# Evaluate over a range of thresholds
thresholds = np.linspace(0.0, 1.0, 101)
tps, fps, fns = [], [], []
precisions, recalls = [], []

for t in thresholds:
    metrics = evaluate_predictions(simulated_predictions, num_gt, t)
    tps.append(metrics["TP"])
    fps.append(metrics["FP"])
    fns.append(metrics["FN"])
    precisions.append(metrics["Precision"])
    recalls.append(metrics["Recall"])

tps = np.array(tps)
fps = np.array(fps)
fns = np.array(fns)
precisions = np.array(precisions)
recalls = np.array(recalls)

In [ ]:
# Plotting the results using Matplotlib
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Precision & Recall vs Threshold
axes[0].plot(thresholds, precisions, label='Precision', color='#1f77b4', linewidth=2.5)
axes[0].plot(thresholds, recalls, label='Recall', color='#ff7f0e', linewidth=2.5)
axes[0].set_xlabel('Confidence Threshold (conf)', fontsize=12)
axes[0].set_ylabel('Metric Value', fontsize=12)
axes[0].set_title('Precision & Recall vs. Threshold', fontsize=14, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend(fontsize=11)
axes[0].set_xlim(0, 1)
axes[0].set_ylim(0, 1.05)

# Plot 2: TP, FP, FN Counts vs Threshold
axes[1].plot(thresholds, tps, label='True Positives (TP)', color='#2ca02c', linewidth=2.5)
axes[1].plot(thresholds, fps, label='False Positives (FP)', color='#d62728', linewidth=2.5)
axes[1].plot(thresholds, fns, label='False Negatives (FN)', color='#9467bd', linewidth=2.5)
axes[1].set_xlabel('Confidence Threshold (conf)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('TP, FP, FN Counts vs. Threshold', fontsize=14, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend(fontsize=11)
axes[1].set_xlim(0, 1)

# Plot 3: Precision-Recall Curve
axes[2].plot(recalls, precisions, color='#8c564b', linewidth=3)
axes[2].set_xlabel('Recall', fontsize=12)
axes[2].set_ylabel('Precision', fontsize=12)
axes[2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].set_xlim(0, 1.05)
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()